<h3>You must download and import this <a href="https://www.kaggle.com/datasets/mldatastudent/league-of-legends-match-data">Kaggle dataset</a>.

<h2>Import requirements

In [1]:
import pandas as pd

<h2>Read in the raw data (per-player)

In [2]:
player_data = pd.read_csv("lol_match_data.csv")
print("Number of columns: ", player_data.shape[1])
print("Number of rows: ", player_data.shape[0])
player_data.head(1)

Number of columns:  108
Number of rows:  205110


,match_matchId,match_gameStartTimestamp,match_gameEndTimestamp,match_gameDuration,match_mapId,match_platformId,player_puuid,player_teamId,player_teamPosition,player_lane,...,player_item3_categories,player_item3_priceTotal,player_item4_name,player_item4_description,player_item4_categories,player_item4_priceTotal,player_item5_name,player_item5_description,player_item5_categories,player_item5_priceTotal
0,LA1_1531159804,1.720820e+12,1.720820e+12,1834.0,11.0,LA1,QPstXBo4FWSoly8yTtzmHFjsgtwUJrVzRhFTWlO3irBaEd...,blue,TOP,JUNGLE,...,"['Health', 'Damage', 'CooldownReduction', 'Abi...",3100,Sterak's Gage,<mainText><stats><attention> 400</attention> H...,"['Health', 'Damage', 'Tenacity']",3200,Stealth Ward,<mainText><stats></stats><br><br> <active>ACTI...,"['Active', 'Jungle', 'Lane', 'Trinket', 'Vision']",0


In [3]:
# this data is gathered from high-ranked lobbies, which contain a small pool of re-ocurring players
# each player may appear more than once in the dataset, on different teams during different matches
# hence, number of rows != number of unique players, 205,110 vs. 24,279
# each game contains 10 players, so we know there are 20,511 total games scraped
print("Number of unique players: ", player_data["player_puuid"].nunique())
print("Number of unique teams: ", len(player_data[["match_matchId", "player_teamId"]].drop_duplicates()))
print("Number of unique matches: ", player_data["match_matchId"].nunique())

Number of unique players:  24279
Number of unique teams:  41022
Number of unique matches:  20511


<h2>Clean and aggregate the data (per-team)

In [4]:
# we only want to keep columns that we will use as features in our model
# we can remove unnecessary columns related to perks and items 
# player_win will eventually be our label (did this team win the game?)
features_of_interest = ["match_matchId", "match_gameDuration", "player_teamId", "player_teamPosition", "player_win",
                          "player_kills", "player_deaths", "player_assists", "player_goldEarned", "player_visionScore",
                          "team_baron_kills", "team_dragon_kills", "team_riftHerald_kills", "team_tower_kills"]
player_data = player_data[features_of_interest]

In [5]:
# here is one team, notice the 5 different player roles
# player-specific stats such as player_kills vary per row
# while team stats such as player_win are the same
player_data.head(5)

,match_matchId,match_gameDuration,player_teamId,player_teamPosition,player_win,player_kills,player_deaths,player_assists,player_goldEarned,player_visionScore,team_baron_kills,team_dragon_kills,team_riftHerald_kills,team_tower_kills
0,LA1_1531159804,1834.0,blue,TOP,True,14.0,4.0,5.0,15613.0,29.0,1.0,3.0,0.0,7.0
1,LA1_1531159804,1834.0,blue,JUNGLE,True,2.0,4.0,17.0,10279.0,28.0,1.0,3.0,0.0,7.0
2,LA1_1531159804,1834.0,blue,MIDDLE,True,2.0,6.0,16.0,10314.0,20.0,1.0,3.0,0.0,7.0
3,LA1_1531159804,1834.0,blue,BOTTOM,True,19.0,7.0,10.0,17195.0,23.0,1.0,3.0,0.0,7.0
4,LA1_1531159804,1834.0,blue,UTILITY,True,2.0,5.0,24.0,9350.0,87.0,1.0,3.0,0.0,7.0


<h3>Isolate position-specific data

In [6]:
top = player_data[player_data["player_teamPosition"] == "TOP"]
jungle = player_data[player_data["player_teamPosition"] == "JUNGLE"]
middle = player_data[player_data["player_teamPosition"] == "MIDDLE"]
bottom = player_data[player_data["player_teamPosition"] == "BOTTOM"]
support = player_data[player_data["player_teamPosition"] == "UTILITY"]